# SKL metric analysis

## Step 1 — Read training and metric results

- `read_training_results(...)` reads the last epoch row from each training log.
- `read_metric_results(...)` reads the first `BeforeRescale` row from each metric log.

Both functions return one flat `pandas.DataFrame` row per `.txt` file.

In [1]:
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

### Filename parsing

The patterns below define the hyperparameters encoded in the filenames. Add another `(column_name, pattern)` entry if a future experiment introduces a new filename field.

In [2]:
NUMBER_PATTERN = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"

# Each regex captures only the value following a known filename prefix.
HYPERPARAM_PATTERNS = (
    ("aug", rf"(?:^|_)aug(?P<value>True|False)(?=_|$)"),
    ("disableNorm", rf"(?:^|_)disableNorm(?P<value>True|False)(?=_|$)"),
    ("opt", r"(?:^|_)opt(?P<value>[^_]+)(?=_|$)"),
    ("epochs", rf"(?:^|_)epochs(?P<value>\d+)(?=_|$)"),
    ("bsize", rf"(?:^|_)bsize(?P<value>\d+)(?=_|$)"),
    ("LR", rf"(?:^|_)LR(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("PWD", rf"(?:^|_)PWD(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    ("WD", rf"(?:^|_)WD(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("FuncWD", rf"(?:^|_)FuncWD(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    ("mom", rf"(?:^|_)mom(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("NJ", rf"(?:^|_)NJ(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    ### `scheduler` and `plateau` share one filename token in current logs.
    # ("scheduler", r"(?:^|_)scheduler(?P<value>True|False)"),
    # ("plateau", rf"plateau(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("rho", rf"(?:^|_)rho(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("adapsam", r"(?:^|_)adapsam(?P<value>True|False)(?=_|$)"),
    # ("labelsm", rf"(?:^|_)labelsm(?P<value>{NUMBER_PATTERN})(?=_|$)"),
)

HYPERPARAM_COLUMNS = [name for name, _ in HYPERPARAM_PATTERNS]

PERFORMANCE_COLUMNS = [
    "epoch_runned",
    "train_loss",
    "train_acc",
    "test_loss",
    "test_acc",
]


def smart_cast(value: str):
    """Convert filename values to bool/int/float when possible."""
    value = value.strip()
    if value in {"True", "False"}:
        return value == "True"
    if re.fullmatch(r"[-+]?\d+", value):
        return int(value)
    if re.fullmatch(NUMBER_PATTERN, value):
        return float(value)
    return value


def parse_hparams_from_filename(log_path):
    """Extract known hyperparameter choices from a training or metric filename."""
    stem = Path(log_path).stem
    hparams = {}
    for column, pattern in HYPERPARAM_PATTERNS:
        match = re.search(pattern, stem)
        if match is not None:
            hparams[column] = smart_cast(match.group("value"))
    return hparams

### Shared log helpers

In [3]:
EPOCH_PATTERN = re.compile(r"^Epoch\s+(\d+)")
EPOCHS_RUN_PATTERN = re.compile(r"^--Epochs run:\s*(\d+)")


def parse_numeric(value: str) -> float:
    """Parse log numbers, including scientific notation, nan/inf, and times ending in `s`."""
    value = value.strip()
    if value.endswith("s"):
        value = value[:-1].strip()
    try:
        return float(value)
    except ValueError as exc:
        raise ValueError(f"Expected a numeric log value, got {value!r}") from exc


def parse_epoch(line: str) -> int:
    match = EPOCH_PATTERN.search(line.strip())
    if match is None:
        raise ValueError(f"Could not parse an epoch from: {line!r}")
    return int(match.group(1))


def parse_pipe_fields(line: str, start_at: int = 1) -> dict:
    """Parse `label: value` fields separated by vertical bars."""
    fields = {}
    for part in line.split("|")[start_at:]:
        part = part.strip()
        if not part or ":" not in part:
            continue
        label, value = part.split(":", 1)
        fields[label.strip()] = parse_numeric(value)
    return fields


def ordered_dataframe(rows: list[dict], leading_columns: list[str]) -> pd.DataFrame:
    """Put identifiers/hyperparameters first and retain every discovered result column."""
    frame = pd.DataFrame(rows)
    leading = [column for column in leading_columns if column in frame.columns]
    remaining = [column for column in frame.columns if column not in leading]
    return frame.loc[:, leading + remaining]


def sort_result_dataframe(frame, sorting_key, sorting_order):
    """Sort result rows by one column using an explicit order."""
    order_to_ascending = {"ascending": True, "descending": False}
    if sorting_order not in order_to_ascending:
        raise ValueError(
            "sorting_order must be either 'ascending' or 'descending'"
        )
    if sorting_key not in frame.columns:
        raise KeyError(
            f"Sorting key {sorting_key!r} is not a result column"
        )

    return frame.sort_values(
        by=sorting_key,
        ascending=order_to_ascending[sorting_order],
        na_position="last",
        kind="stable",
    ).reset_index(drop=True)


def collect_result_rows(log_dir, parse_file, pattern="*.txt", strict=False):
    """Apply a single-file parser to all matching files in a directory."""
    log_dir = Path(log_dir)
    files = sorted(log_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern!r} in {log_dir}")

    rows = []
    for filepath in files:
        try:
            rows.append(parse_file(filepath))
        except (OSError, ValueError, KeyError) as exc:
            if strict:
                raise RuntimeError(f"Failed to parse {filepath}") from exc
            warnings.warn(f"Skipping {filepath}: {exc}", stacklevel=2)
    if not rows:
        raise RuntimeError(f"No valid result rows were parsed from {log_dir}")
    return rows

### Training-result reader

`Performance' values and `final_LR` come from the last `Epoch ...` row. 

`epoch_runned` uses the explicit '--Epochs run:' footer when present; otherwise it is the zero-based last epoch index plus one.

In [4]:
TRAINING_FIELD_MAP = {
    "LR": "final_LR",
    "Train Loss": "train_loss",
    "Train Acc": "train_acc",
    "Test Loss": "test_loss",
    "Test Acc": "test_acc",
}


def parse_training_result_file(log_path) -> dict:
    """Parse one training log into one flat result dictionary."""
    log_path = Path(log_path)
    last_epoch_line = None
    epochs_runned = None

    with log_path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if EPOCH_PATTERN.match(line):
                last_epoch_line = line
            footer_match = EPOCHS_RUN_PATTERN.match(line)
            if footer_match is not None:
                epochs_runned = int(footer_match.group(1))

    if last_epoch_line is None:
        raise ValueError("No epoch logging row was found")

    fields = parse_pipe_fields(last_epoch_line)
    missing = [label for label in TRAINING_FIELD_MAP if label not in fields]
    if missing:
        raise ValueError(f"Last epoch row is missing fields: {missing}")

    # Epoch indices are zero-based, while epoch_runned is a count.
    if epochs_runned is None:
        epochs_runned = parse_epoch(last_epoch_line) + 1

    row = {"filepath": str(log_path)}
    row.update(parse_hparams_from_filename(log_path))
    row["epoch_runned"] = epochs_runned
    for log_label, column in TRAINING_FIELD_MAP.items():
        row[column] = fields[log_label]
    return row


def read_training_results(
    log_dir, pattern="*.txt", strict=False,
    sorting_key="test_loss", sorting_order="descending",
) -> pd.DataFrame:
    """Read and sort all training logs into one row-per-file DataFrame."""
    rows = collect_result_rows(
        log_dir, parse_training_result_file, pattern=pattern, strict=strict
    )
    leading = ["filepath", 
               *HYPERPARAM_COLUMNS, 
               *PERFORMANCE_COLUMNS, 
               "final_LR"]
    frame = ordered_dataframe(rows, leading)
    return sort_result_dataframe(frame, sorting_key, sorting_order)

### Metric-result reader

Only the first `BeforeRescale` row is used. 

The five requested performance fields receive standardized names; 

every other labeled value retains its exact log label. 

Consequently, a field such as 'KL_norm_uniform: nan' still creates a 'KL_norm_uniform' column containing 'NaN'.

In [5]:
METRIC_PERFORMANCE_MAP = {
    "Train Loss": "train_loss",
    "Train Acc": "train_acc",
    "Test Loss": "test_loss",
    "Test Acc": "test_acc",
}


def parse_metric_result_file(log_path) -> dict:
    """Parse the first BeforeRescale row in one metric-result log."""
    log_path = Path(log_path)
    before_rescale_line = None

    with log_path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            parts = [part.strip() for part in line.split("|")]
            if EPOCH_PATTERN.match(line) and len(parts) > 1 and parts[1] == "BeforeRescale":
                before_rescale_line = line
                break

    if before_rescale_line is None:
        raise ValueError("No BeforeRescale row was found")

    # Skip both the epoch and status fields; remaining fields are label/value pairs.
    fields = parse_pipe_fields(before_rescale_line, start_at=2)
    missing = [label for label in METRIC_PERFORMANCE_MAP if label not in fields]
    if missing:
        raise ValueError(f"BeforeRescale row is missing fields: {missing}")

    row = {"filepath": str(log_path)}
    row.update(parse_hparams_from_filename(log_path))
    row["epoch_runned"] = parse_epoch(before_rescale_line)

    # Standardize performance names and preserve every other log label verbatim.
    for log_label, value in fields.items():
        column = METRIC_PERFORMANCE_MAP.get(log_label, log_label)
        row[column] = value
    return row


def read_metric_results(
    log_dir, pattern="*.txt", strict=False,
    sorting_key="test_loss", sorting_order="descending",
) -> pd.DataFrame:
    """Read and sort all metric logs into one row-per-file DataFrame."""
    rows = collect_result_rows(
        log_dir, parse_metric_result_file, pattern=pattern, strict=strict
    )
    leading = ["filepath", 
               *HYPERPARAM_COLUMNS, 
               *PERFORMANCE_COLUMNS]
    frame = ordered_dataframe(rows, leading)
    return sort_result_dataframe(frame, sorting_key, sorting_order)

### Load all model results

The path setup works whether Jupyter starts in this notebook's directory or at the workspace root.

In [6]:
cwd = Path.cwd()
if cwd.name == "result-RegressionPCR":
    project_root = cwd.parent
elif (cwd / "project_SCI").is_dir():
    project_root = cwd / "project_SCI"
else:
    raise FileNotFoundError("Run this notebook from the workspace root or result-RegressionPCR directory.")

result_root = project_root / "result_trainedmodel"


# DATA~MODEL
DATASET_MODELS = {
    "cifar10": [
        "resnet18_BN", "vgg13_BN",
        "resnet18_noBN", "vgg13_noBN",
    ],
    "cifar100": [
        "wideresnetpostact_BN", "wideresnetpostact_noBN",
        "vit_BN", "vit_noBN",
    ],
}

# Keep a flat model list for the model-wise loops in later sections.
MODELS = [
    model_name
    for dataset_models in DATASET_MODELS.values()
    for model_name in dataset_models
]

# MODEL~DATA
MODEL_DATASETS = {
    model_name: dataset
    for dataset, dataset_models in DATASET_MODELS.items()
    for model_name in dataset_models
}

SORTING_KEY = "test_loss"
SORTING_ORDER = "descending"

METRIC_txt_PATTERM = "newtrained31*.txt" #<<<<<<<<<<<<<<<<<<<<<<


# Read every directory once and retain dictionaries for convenient iteration.
training_dfs = {}
metric_dfs = {}
load_records = []
for dataset, dataset_models in DATASET_MODELS.items():
    for model_name in dataset_models:

        training_dfs[model_name] = read_training_results(
            result_root / dataset / model_name, pattern="*.txt",
            sorting_key=SORTING_KEY, sorting_order=SORTING_ORDER,
        )
        metric_dfs[model_name] = read_metric_results(
            result_root / dataset / f"{model_name}_metrics", pattern=METRIC_txt_PATTERM, #<<<<<<<<<<<<<<<<<<<<<<
            sorting_key=SORTING_KEY, sorting_order=SORTING_ORDER,
        )

        load_records.append(
            {
                "dataset": dataset,
                "model": model_name,
                "training_shape": training_dfs[model_name].shape,
                "metric_shape": metric_dfs[model_name].shape,
            }
        )

        print(f"Dataset: {dataset} | Model: {model_name}")

        # Training results
        display(training_dfs[model_name].head(10))

        # Metric results
        # display(metric_dfs[model_name].head(5))

load_summary = pd.DataFrame(load_records)
display(load_summary)


# Explicit names make individual model results 
# convenient to use in later cells.
resnet18_BN_training_df = training_dfs["resnet18_BN"]
resnet18_BN_metric_df = metric_dfs["resnet18_BN"]
resnet18_noBN_training_df = training_dfs["resnet18_noBN"]
resnet18_noBN_metric_df = metric_dfs["resnet18_noBN"]

vgg13_BN_training_df = training_dfs["vgg13_BN"]
vgg13_BN_metric_df = metric_dfs["vgg13_BN"]
vgg13_noBN_training_df = training_dfs["vgg13_noBN"]
vgg13_noBN_metric_df = metric_dfs["vgg13_noBN"]

wideresnetpostact_BN_training_df = training_dfs["wideresnetpostact_BN"]
wideresnetpostact_BN_metric_df = metric_dfs["wideresnetpostact_BN"]
wideresnetpostact_noBN_training_df = training_dfs["wideresnetpostact_noBN"]
wideresnetpostact_noBN_metric_df = metric_dfs["wideresnetpostact_noBN"]

vit_BN_training_df = training_dfs["vit_BN"]
vit_BN_metric_df = metric_dfs["vit_BN"]
vit_noBN_training_df = training_dfs["vit_noBN"]
vit_noBN_metric_df = metric_dfs["vit_noBN"]

Dataset: cifar10 | Model: resnet18_BN


,filepath,aug,disableNorm,opt,epochs,bsize,LR,WD,mom,epoch_runned,train_loss,train_acc,test_loss,test_acc,final_LR
0,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,5000,6,93,2.3026,10.0,2.3026,10.00,0.000100
1,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,5000,9,87,2.3026,10.0,2.3026,10.00,0.000100
2,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,5000,5000,9,88,2.3026,10.0,2.3026,10.00,0.000050
3,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,10,6,102,0.0006,100.0,1.3776,68.05,0.000001
4,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,50,6,101,0.0007,100.0,1.3608,68.60,0.000001
5,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,100,6,102,0.0008,100.0,1.3310,68.43,0.000001
6,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,500,6,101,0.0016,100.0,1.1887,68.65,0.000001
7,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,1000,6,102,0.0031,100.0,1.0893,69.04,0.000001
8,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,10,9,94,0.0002,100.0,1.0277,75.86,0.000001
9,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,50,9,94,0.0002,100.0,0.9533,76.63,0.000001


Dataset: cifar10 | Model: vgg13_BN


,filepath,aug,disableNorm,opt,epochs,bsize,LR,WD,mom,epoch_runned,train_loss,train_acc,test_loss,test_acc,final_LR
0,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,5000,6,92,2.3026,10.00,2.3026,10.00,1.000000e-04
1,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,5000,9,85,2.3026,10.00,2.3026,10.00,1.000000e-04
2,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,5000,5000,9,85,2.3026,10.00,2.3026,10.00,5.000000e-05
3,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,10,6,93,0.0007,100.00,1.1504,73.40,1.000000e-06
4,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,50,6,93,0.0007,100.00,1.1441,72.42,1.000000e-06
5,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,100,6,93,0.0008,100.00,1.1379,73.16,1.000000e-06
6,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,500,6,93,0.0014,100.00,1.0545,73.01,1.000000e-06
7,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,1000,9,800,0.3118,91.35,0.9898,70.48,1.000000e-08
8,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,1000,6,93,0.0027,100.00,0.9482,73.39,1.000000e-06
9,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,100,10,9,94,0.0001,100.00,0.8993,80.71,1.000000e-06


Dataset: cifar10 | Model: resnet18_noBN


...

Dataset: cifar100 | Model: vit_BN


,filepath,aug,disableNorm,opt,epochs,bsize,LR,WD,mom,epoch_runned,train_loss,train_acc,test_loss,test_acc,final_LR
0,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,500,500,0,629,0.0003,99.98,11.2964,16.56,9.765600e-06
1,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,500,800,0,508,0.0003,99.98,10.0127,19.07,9.765600e-06
2,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,500,1000,0,550,0.0003,99.98,8.8618,21.74,9.765600e-06
3,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,200,800,0,369,0.0003,99.98,8.4078,26.76,3.906300e-06
4,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,200,500,0,419,0.0003,99.98,8.2041,30.34,3.906300e-06
5,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,200,1000,0,340,0.0003,99.98,7.7610,31.76,3.906300e-06
6,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,100,500,0,360,0.0003,99.98,6.6364,38.88,1.953100e-06
7,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,100,800,0,323,0.0003,99.98,6.5175,38.08,1.953100e-06
8,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,100,1000,0,321,0.0003,99.98,6.3930,38.16,1.953100e-06
9,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,50,500,0,296,0.0003,99.98,6.1064,40.64,9.765600e-07


Dataset: cifar100 | Model: vit_noBN


,filepath,aug,disableNorm,opt,epochs,bsize,LR,WD,mom,epoch_runned,train_loss,train_acc,test_loss,test_acc,final_LR
0,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,50000,0,302,0.0004,99.98,7.2588,34.37,1.953100e-07
1,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,80000,0,340,0.0005,99.98,7.2515,33.83,1.953100e-07
2,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,1000,0,322,0.0003,99.98,6.9620,40.64,1.953100e-07
3,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,10000,0,275,0.0003,99.98,6.9514,38.85,1.953100e-07
4,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,500,0,288,0.0003,99.98,6.7581,40.36,1.953100e-07
5,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,8000,0,304,0.0003,99.98,6.7577,40.21,1.953100e-07
6,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,800,0,306,0.0003,99.98,6.7346,40.40,1.953100e-07
7,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,5000,0,324,0.0003,99.98,6.6262,40.75,1.953100e-07
8,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,20,80000,0,347,0.0004,99.98,6.1688,40.51,3.906300e-07
9,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,20,10000,0,280,0.0003,99.98,6.1579,45.06,3.906300e-07


,dataset,model,training_shape,metric_shape
0,cifar10,resnet18_BN,"(48, 15)","(48, 53)"
1,cifar10,vgg13_BN,"(48, 15)","(48, 53)"
2,cifar10,resnet18_noBN,"(60, 15)","(60, 53)"
3,cifar10,vgg13_noBN,"(60, 15)","(60, 53)"
4,cifar100,wideresnetpostact_BN,"(48, 15)","(48, 53)"
5,cifar100,wideresnetpostact_noBN,"(48, 15)","(48, 53)"
6,cifar100,vit_BN,"(48, 15)","(48, 53)"
7,cifar100,vit_noBN,"(48, 15)","(48, 53)"


## Observation:
1) more than half of ViT and noLN-ViT trained models severely overfit, with huge test_loss values.

2) noBN-Wideresnet also overfit.

⬇️------------------
For debug usage. Inpect the dataframe, no need to run this subsection everytime.

In [7]:
# # For Debug
# ACTIVE_MODEL = "wideresnetpostact_noBN"  #"resnet18_BN"

# training_df = training_dfs[ACTIVE_MODEL]
# metric_df = metric_dfs[ACTIVE_MODEL]

# metric_df.columns

# # Inspect only computed metrics, excluding paths, hyperparameters, and performance fields.
# non_metric_columns = {"filepath", *HYPERPARAM_COLUMNS, *PERFORMANCE_COLUMNS}
# metric_columns = [
#     column for column in metric_df.columns
#     if column not in non_metric_columns
# ]

# # Confirm that fields logged as `nan` remain present as DataFrame columns.
# nan_metric_columns = [
#     column for column in metric_columns
#     if metric_df[column].isna().any()]
# print("Metric columns containing at least one NaN:")
# print(nan_metric_columns)

# # Identify metrics that are zero or negative for at least one run.
# nonpositive_metric_columns = [
#     column for column in metric_columns
#     if metric_df[column].le(0).any()]
# print("\nMetric columns containing at least one value <= 0:")
# print(nonpositive_metric_columns)

# # Identify metrics that are strictly negative for at least one run.
# negative_metric_columns = [
#     column for column in metric_columns
#     if metric_df[column].lt(0).any()]
# print("\nMetric columns containing at least one value < 0:")
# print(negative_metric_columns)

⬆️------------------

## Step 2 — Prepare data for analysis

### Step 2.1 — Add derived columns

In [8]:
def add_derived_metric_columns(metric_df, Eloss_sigma=0.01):
    """Return a copy with the derived columns used by later analyses."""
    ready_df = metric_df.copy()

    ready_df["gap"] = ready_df["test_loss"] - ready_df["train_loss"]
    ready_df["KL(og)_sqrt"] = np.sqrt(ready_df["KL(og)"])

    ready_df["KL_func_C_aniso_sigmaOne"] = (ready_df["KL_func_C_aniso"] * (Eloss_sigma ** 2))
    ready_df["KL_func_C_iso_sigmaOne"] = (ready_df["KL_func_C_iso"] * (Eloss_sigma ** 2))
    return ready_df


# Prepare every model while keeping the parsed metric DataFrames unchanged.
metric_dfs_readytogo = {
    model_name: add_derived_metric_columns(metric_dfs[model_name])
    for model_name in MODELS
}

resnet18_BN_metric_df_readytogo = metric_dfs_readytogo["resnet18_BN"]
vgg13_BN_metric_df_readytogo = metric_dfs_readytogo["vgg13_BN"]
resnet18_noBN_metric_df_readytogo = metric_dfs_readytogo["resnet18_noBN"]
vgg13_noBN_metric_df_readytogo = metric_dfs_readytogo["vgg13_noBN"]
wideresnetpostact_BN_metric_df_readytogo = (metric_dfs_readytogo["wideresnetpostact_BN"])
wideresnetpostact_noBN_metric_df_readytogo = (metric_dfs_readytogo["wideresnetpostact_noBN"])
vit_BN_metric_df_readytogo = (metric_dfs_readytogo["vit_BN"])
vit_noBN_metric_df_readytogo = (metric_dfs_readytogo["vit_noBN"])

print(
    "Added derived columns: \n"
    "  gap, KL(og)_sqrt, KL_func_C_aniso_sigmaOne, and KL_func_C_iso_sigmaOne.\n"
    "    ==> DataFrames 'modelXX_metric_df_readytogo' are ready to go!"
)
# for model_name in MODELS:
#     print(f"Model: {model_name}")
#     display(metric_dfs_readytogo[model_name][["gap", "KL(og)_sqrt"]].head())

Added derived columns: 
  gap, KL(og)_sqrt, KL_func_C_aniso_sigmaOne, and KL_func_C_iso_sigmaOne.
    ==> DataFrames 'modelXX_metric_df_readytogo' are ready to go!


In [9]:
vit_BN_metric_df_readytogo.columns

Index(['filepath', 'aug', 'disableNorm', 'opt', 'epochs', 'bsize', 'LR', 'WD',
       'mom', 'epoch_runned', 'train_loss', 'train_acc', 'test_loss',
       'test_acc', 'S(og)', 'KL(og)', 'KL_spec_norm_prod',
       'KL_spec_norm_prod_n_root', 'KL_path_norm', 'KL_path_norm_n_root',
       'KL_norm_uniform', 'S_trace_uniform', 'KL_norm_minL2', 'S_trace_minL2',
       'S_adap_sam', 'S_Eloss_aniso', 'S_Eloss_iso', 'kernelS_shared_alpha1',
       'kernelS_shared_alpha2', 'kernelS_shared_alpha3',
       'kernelS_indep_alpha1', 'kernelS_indep_alpha2', 'kernelS_indep_alpha3',
       'inputS_alpha1', 'inputS_alpha2', 'inputS_alpha3', 'KL_func_det',
       'KL_func_det_centered', 'KL_func_det_unsquare_centered',
       'KL_func_det_avg', 'KL_func_det_centered_avg',
       'KL_func_det_unsquare_centered_avg', 'KL_func_aniso',
       'KL_func_AB_aniso', 'KL_func_C_aniso', 'KL_func_C2_aniso',
       'KL_func_C3_aniso', 'KL_func_iso', 'KL_func_AB_iso', 'KL_func_C_iso',
       'KL_func_C2_iso', 'KL_f

### Step 2.2 — Define sharpness-complexity metric pairs

`SC_metric_pairs` stores exact DataFrame column names, 

`SC_metric_pair_labels` stores labels for PCR plots, and 

`single_metrics` stores the unique original column names used later for single-factor regression. 

Change 'METRIC_EXT' to switch between anisotropic and isotropic functional metrics.

In [10]:
def build_SC_metric_catalog(ext="aniso"):
    """Build metric pairs and plot labels for one functional-metric variant."""
    if ext not in {"aniso", "iso"}:
        raise ValueError("ext must be either 'aniso' or 'iso'")

    eloss_metric = f"S_Eloss_{ext}"
    func_c_metric = f"KL_func_C_{ext}_sigmaOne"

    # ===============
    SC_metric_pairs = {
        "pair_original": ("S(og)", "KL(og)"),
        "pair_original_sqrt": ("S(og)", "KL(og)_sqrt"),
        "pair_adapsam_spec_norm": ("S_adap_sam", "KL_spec_norm_prod"),
        "pair_adapsam_spec_norm_n_root": ("S_adap_sam", "KL_spec_norm_prod_n_root"),
        "pair_adapsam_path_norm": ("S_adap_sam", "KL_path_norm"),
        "pair_adapsam_path_norm_n_root": ("S_adap_sam", "KL_path_norm_n_root"),
        "pair_adapsam_func_c_sigma_one": ("S_adap_sam", func_c_metric),
        "pair_adapsam_func_det_unsquare_centered": ("S_adap_sam", "KL_func_det_unsquare_centered"),
        "pair_bayesS_func_c_sigma_one":  (eloss_metric, func_c_metric),
        "pair_bayesS_func_det_unsquare_centered": (eloss_metric, "KL_func_det_unsquare_centered"),
    }

    # Generate the repeated three-alpha families without duplicating definitions.
    alpha_indices = (1, 2, 3)

    for direction in ("shared", "indep"):
        for alpha in alpha_indices:
            pair_name = f"pair_kernel_{direction}_alpha{alpha}_func_det_unsquare_centered"
            sharpness = f"kernelS_{direction}_alpha{alpha}"

            SC_metric_pairs[pair_name] = (sharpness, "KL_func_det_unsquare_centered")
            #e.g., "pair_kernel_shared_alpha1_func_det_unsquare_centered": ("kernelS_shared_alpha1", "KL_func_det_unsquare_centered")

    for alpha in alpha_indices:
        pair_name = f"pair_input_alpha{alpha}_func_det_unsquare_centered"
        input_sharpness = f"inputS_alpha{alpha}"
        SC_metric_pairs[pair_name] = (input_sharpness, "KL_func_det_unsquare_centered")
        #e.g., 
        # "pair_input_alpha1_func_det_unsquare_centered": ("inputS_alpha1", "KL_func_det_unsquare_centered")


    metric_plot_labels = {
        "S(og)": "traceH",
        "KL(og)": "l2norm_sq",
        "KL(og)_sqrt": "l2norm",
        "S_adap_sam": "adapSAM",
        "KL_spec_norm_prod": "spec_norm_prod",
        "KL_spec_norm_prod_n_root": "spec_norm_prod", # Note: in the plot legend label, does not show n-root
        "KL_path_norm": "path_norm",
        "KL_path_norm_n_root": "path_norm", # Note: in the plot legend label, does not show n-root
        eloss_metric: f"bayesS_{ext}",
        func_c_metric: f"funcKL_{ext}",
        "KL_func_det_unsquare_centered": "func_norm", #_unsquared
    }
    for direction in ("shared", "indep"):
        for alpha in alpha_indices:
            # NAMING <<<
            metric_plot_labels[f"kernelS_{direction}_alpha{alpha}"] = (
                                                                f"kernelS_{direction}_alpha{alpha}"
                                                                )
    for alpha in alpha_indices:
        metric_plot_labels[f"inputS_alpha{alpha}"] = (
            f"inputS_alpha{alpha}"
        )

    # ===============
    SC_metric_pair_labels = {
        pair_name: f"({metric_plot_labels[s_metric]}, {metric_plot_labels[c_metric]})"
        for pair_name, (s_metric, c_metric) in SC_metric_pairs.items()
    }

    # ===============
    # Keep unique original column names in their first-appearance order.
    single_metrics = list(
        dict.fromkeys(
            metric
            for pair in SC_metric_pairs.values()
            for metric in pair
        )
    )

    return SC_metric_pairs, SC_metric_pair_labels, single_metrics


METRIC_EXT = "aniso"
SC_metric_pairs, SC_metric_pair_labels, single_metrics = build_SC_metric_catalog(ext=METRIC_EXT)

# Fail early if a selected metric is unavailable for any model.
required_metric_columns = set(single_metrics)
for model_name, ready_df in metric_dfs_readytogo.items():
    missing_columns = required_metric_columns.difference(ready_df.columns)
    if missing_columns:
        raise KeyError(f"{model_name} is missing metric columns: {sorted(missing_columns)}")

print(
    f"Prepared "
    f"{len(SC_metric_pairs)} S-C pairs and "
    f"{len(single_metrics)} unique single metrics for ext={METRIC_EXT!r}."
)

display(
    pd.DataFrame(
        {"columns": SC_metric_pairs, 
         "plot_label": SC_metric_pair_labels}
    ).rename_axis("pair_name")
)
display(
    pd.DataFrame({"metric": single_metrics})
    )

Prepared 19 S-C pairs and 20 unique single metrics for ext='aniso'.


,columns,plot_label
pair_name,,
pair_original,"(S(og), KL(og))","(traceH, l2norm_sq)"
pair_original_sqrt,"(S(og), KL(og)_sqrt)","(traceH, l2norm)"
pair_adapsam_spec_norm,"(S_adap_sam, KL_spec_norm_prod)","(adapSAM, spec_norm_prod)"
pair_adapsam_spec_norm_n_root,"(S_adap_sam, KL_spec_norm_prod_n_root)","(adapSAM, spec_norm_prod)"
pair_adapsam_path_norm,"(S_adap_sam, KL_path_norm)","(adapSAM, path_norm)"
pair_adapsam_path_norm_n_root,"(S_adap_sam, KL_path_norm_n_root)","(adapSAM, path_norm)"
pair_adapsam_func_c_sigma_one,"(S_adap_sam, KL_func_C_aniso_sigmaOne)","(adapSAM, funcKL_aniso)"
pair_adapsam_func_det_unsquare_centered,"(S_adap_sam, KL_func_det_unsquare_centered)","(adapSAM, func_norm)"
pair_bayesS_func_c_sigma_one,"(S_Eloss_aniso, KL_func_C_aniso_sigmaOne)","(bayesS_aniso, funcKL_aniso)"


,metric
0,S(og)
1,KL(og)
2,KL(og)_sqrt
3,S_adap_sam
4,KL_spec_norm_prod
5,KL_spec_norm_prod_n_root
6,KL_path_norm
7,KL_path_norm_n_root
8,KL_func_C_aniso_sigmaOne
9,KL_func_det_unsquare_centered


## Step 3 — Regression and Pareto analysis

### Step 3.1 — Analysis helpers

PCR is the fraction of comparable point pairs for which 
the point with no larger S and C has worse target performance. 
Thus, a smaller PCR indicates better agreement between the metric pair and performance.

In [11]:
def filter_well_trained_points(df, train_loss_max=None, test_loss_max=None):
    """Return rows satisfying the enabled train/test loss upper bounds."""
    required = [
        column
        for column, threshold in (
            ("train_loss", train_loss_max),
            ("test_loss", test_loss_max),
        )
        if threshold is not None and column not in df.columns
    ]
    if required:
        raise KeyError(f"Missing training-quality columns: {required}")

    filtered = df.copy()
    mask = pd.Series(True, index=filtered.index, dtype=bool)
    descriptions = []

    if train_loss_max is not None:
        filtered["train_loss"] = pd.to_numeric(filtered["train_loss"], errors="coerce")
        mask &= filtered["train_loss"].notna()
        mask &= filtered["train_loss"] <= train_loss_max
        descriptions.append(f"train_loss <= {train_loss_max}")

    if test_loss_max is not None:
        filtered["test_loss"] = pd.to_numeric(filtered["test_loss"], errors="coerce")
        mask &= filtered["test_loss"].notna()
        mask &= filtered["test_loss"] <= test_loss_max
        descriptions.append(f"test_loss <= {test_loss_max}")

    filter_description = " and ".join(descriptions) if descriptions else "no filter"
    return filtered.loc[mask].copy(), filter_description


def select_valid_SC_points(df, s_metric, c_metric, target):
    """Keep finite rows with strictly positive S and C values."""
    required = [s_metric, c_metric, target]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(f"Missing analysis columns: {missing}")

    columns = list(dict.fromkeys(["filepath", *required]))
    columns = [column for column in columns if column in df.columns]
    pair_df = df.loc[:, columns].copy()
    for column in required:
        pair_df[column] = pd.to_numeric(pair_df[column], errors="coerce")

    valid_SC_mask = (
        np.isfinite(pair_df[s_metric])
        & np.isfinite(pair_df[c_metric])
        & (pair_df[s_metric] > 0.0)
        & (pair_df[c_metric] > 0.0)
    )
    return pair_df.loc[valid_SC_mask].copy()


def fit_SC_linear_regression(valid_SC_df, s_metric, c_metric, target):
    """Fit 
            target = intercept + beta_S*S + beta_C*C 
        and return 
            R^2 and coefficients.
    """
    regression_df = (
        valid_SC_df[[s_metric, c_metric, target]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    if len(regression_df) < 3:
        return np.nan, None

    X = regression_df[[s_metric, c_metric]]
    y = regression_df[target]

    model = LinearRegression()
    model.fit(X, y)

    r_squared = model.score(X, y)
    coefficient = f"({model.coef_[0]:.4e}, {model.coef_[1]:.4e})"
    return float(r_squared), coefficient


def compute_PCR(
    valid_SC_df, s_metric, c_metric, 
    target, performance_higher_better=False
):
    """Compute the discrepant/comparable pair ratio used as PCR."""
    pcr_df = (
        valid_SC_df[[s_metric, c_metric, target]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    rows = pcr_df.to_numpy(dtype=float)
    comparable_pairs = 0
    discrepant_pairs = 0

    for i in range(len(rows)):
        a_s, a_c, a_performance = rows[i]
        for j in range(i + 1, len(rows)):
            b_s, b_c, b_performance = rows[j]
            a_dominates_b = a_s <= b_s and a_c <= b_c
            b_dominates_a = b_s <= a_s and b_c <= a_c

            if a_dominates_b:
                comparable_pairs += 1
                worse = (
                    a_performance < b_performance
                    if performance_higher_better
                    else a_performance > b_performance
                )
                discrepant_pairs += int(worse)
            elif b_dominates_a:
                comparable_pairs += 1
                worse = (
                    b_performance < a_performance
                    if performance_higher_better
                    else b_performance > a_performance
                )
                discrepant_pairs += int(worse)

    pcr = discrepant_pairs / comparable_pairs if comparable_pairs else np.nan
    return float(pcr) if np.isfinite(pcr) else np.nan


def compute_pareto_front(valid_SC_df, s_metric, c_metric):
    """Return points not dominated when both S and C are minimized."""
    if valid_SC_df.empty:
        return valid_SC_df.copy()

    values = valid_SC_df[[s_metric, c_metric]].to_numpy(dtype=float)
    dominated = np.zeros(len(values), dtype=bool)
    for i, point in enumerate(values):
        weakly_better = np.all(values <= point, axis=1)
        strictly_better = np.any(values < point, axis=1)
        dominated[i] = np.any(weakly_better & strictly_better)

    return (
        valid_SC_df.loc[~dominated]
        .sort_values([s_metric, c_metric])
        .copy()
    )


def parse_SC_pair_label(pair_label):
    """Return the S and C display labels stored in `(S label, C label)`."""
    label_text = str(pair_label).strip()
    if label_text.startswith("(") and label_text.endswith(")"):
        label_text = label_text[1:-1]
    labels = [label.strip() for label in label_text.split(",", maxsplit=1)]
    if len(labels) != 2 or not all(labels):
        raise ValueError(
            f"pair_label must have the form '(S label, C label)', got {pair_label!r}"
        )
    return labels[0], labels[1]


def format_log_tick(value, position=None):
    """Format exact powers of ten as `1e-1`, `1e0`, `1e1`, and so on."""
    if value <= 0.0 or not np.isfinite(value):
        return ""
    exponent = int(np.round(np.log10(value)))
    if not np.isclose(value, 10.0 ** exponent):
        return ""
    return f"1e{exponent}"


def add_slim_colorbar(
    fig, ax, mappable, title, width="2%", pad="2%", tick_labelsize=7,
):
    """Add a slim colorbar to the right and place its title on top."""
    divider = make_axes_locatable(ax)
    colorbar_ax = divider.append_axes("right", size=width, pad=pad)
    colorbar = fig.colorbar(mappable, cax=colorbar_ax)
    colorbar.ax.set_title(title, fontsize=tick_labelsize+1, pad=3)
    colorbar.ax.tick_params(labelsize=tick_labelsize)
    return colorbar


def plot_PCR_pareto(
    valid_SC_df, s_metric, c_metric, target, pair_label, model_name, pcr,
    log_scale=True, figsize=(5,4),
    markersize=10, alpha=0.80, cmap="viridis",
    colorbar_width="2%", colorbar_pad="2%",
    tick_labelsize=8, annotation_fontsize=7,
    show_annotation_arrows=False,
):
    """
    Plot valid S-C points and their lower-left Pareto front.
    """
    plot_df = (
        valid_SC_df.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[target])
        .copy()
    )
    pareto_df = compute_pareto_front(plot_df, s_metric, c_metric)
    fig, ax = plt.subplots(figsize=figsize)
    x_axis_label, y_axis_label = parse_SC_pair_label(pair_label)
    annotation_texts = []

    if plot_df.empty:
        ax.text(0.5, 0.5, "No valid S-C points", ha="center", va="center")
    else:
        pcr_text = f"{pcr:.1%}" if np.isfinite(pcr) else "N/A"
        scatter = ax.scatter(
            plot_df[s_metric], plot_df[c_metric],
            c=plot_df[target], cmap=cmap,
            s=markersize, alpha=alpha,
            label=f"{pair_label} PCR={pcr_text}"
        )
        add_slim_colorbar(
            fig, ax, scatter, title=target,
            width=colorbar_width, pad=colorbar_pad,
            tick_labelsize=tick_labelsize,
        )

        # The configured target is `gap`; annotate each point by its value.
        for x_value, y_value, gap_value in zip(
            plot_df[s_metric], plot_df[c_metric], plot_df[target]
        ):
            annotation_texts.append(
                ax.text(
                    x_value, y_value, f"{gap_value:.2f}",
                    fontsize=annotation_fontsize,
                )
            )
        # Match the Pareto line to the selected colormap.
        pareto_line_color = scatter.cmap(0.30)
        ax.plot(
            pareto_df[s_metric], pareto_df[c_metric],
            color=pareto_line_color, linewidth=0.9,
            label="Pareto front",
        )
        ax.legend(fontsize=annotation_fontsize+1.0)

        if log_scale:
            ax.set_xscale("log")
            ax.set_yscale("log")
            # Use plain scientific notation (for example, 1e1) on both axes.
            for axis in (ax.xaxis, ax.yaxis):
                axis.set_major_locator(
                    ticker.LogLocator(base=10.0, subs=(1.0,))
                )
                axis.set_major_formatter(ticker.FuncFormatter(format_log_tick))
                axis.set_minor_formatter(ticker.NullFormatter())

    if annotation_texts:
        adjust_kwargs = {"ax": ax}
        if show_annotation_arrows:
            adjust_kwargs["arrowprops"] = {
                "arrowstyle": "-", "color": "0.4",
                "linewidth": 0.5, "alpha": 0.7,
            }
        adjust_text(annotation_texts, **adjust_kwargs)

    # ax.set_title(f"{model_name}: {pair_label}")
    ax.set_xlabel(f"S metric {x_axis_label}")
    ax.set_ylabel(f"C metric {y_axis_label}")
    ax.tick_params(axis="both", which="both", labelsize=tick_labelsize)
    ax.grid(True, linestyle="--", alpha=0.3)
    fig.tight_layout()
    plt.show()
    return fig, ax


#========== The Engine ==========
def analyze_metric_dataframe(
    metric_df, model_name, SC_metric_pairs, SC_metric_pair_labels,
    target="gap", train_loss_max=None, test_loss_max=None,
    performance_higher_better=False,
    sort_summary_table_by=None,
    plot_PCR=False, 
    figsize=(5,4),
    log_scale=True, markersize=10, alpha=0.80,
    cmap="viridis", colorbar_width="5%", colorbar_pad="3%",
    tick_labelsize=9, annotation_fontsize=7,
    show_annotation_arrows=False,
):
    """
    Run filtering, PCR, regression, and 
    optional Pareto plots for one model.
    """
    if target not in metric_df.columns:
        raise KeyError(f"Target column {target!r} is unavailable for {model_name}")
    if sort_summary_table_by not in {None, "PCR", "R^2"}:
        raise ValueError("sort_summary_table_by must be None, 'PCR', or 'R^2'")

    well_trained_df, filter_description = filter_well_trained_points(
        metric_df,
        train_loss_max=train_loss_max,
        test_loss_max=test_loss_max,
    )
    print(
        f"{model_name}: {len(well_trained_df)}/{len(metric_df)} well-trained points "
        f"({filter_description})."
    )

    records = []
    valid_data_by_pair = {}
    for pair_name, (s_metric, c_metric) in SC_metric_pairs.items():
        valid_SC_df = select_valid_SC_points(
            well_trained_df, s_metric, c_metric, target
        )
        valid_data_by_pair[pair_name] = valid_SC_df
        pcr = compute_PCR(
            valid_SC_df, s_metric, c_metric, target,
            performance_higher_better=performance_higher_better,
        )
        r_squared, coefficient = fit_SC_linear_regression(
            valid_SC_df, s_metric, c_metric, target
        )
        records.append(
            {
                "pair_name": pair_name,
                "metric_pair": SC_metric_pair_labels[pair_name],
                "valid_SC_points": len(valid_SC_df),
                "PCR": pcr,
                "R^2": r_squared,
                "coefficient": coefficient,
            }
        )

    summary = pd.DataFrame(records).set_index("pair_name")
    summary = summary[
        ["metric_pair", "valid_SC_points", "PCR", "R^2", "coefficient"]
    ]
    if sort_summary_table_by == "PCR":
        summary = summary.sort_values("PCR", ascending=True, na_position="last")
    elif sort_summary_table_by == "R^2":
        summary = summary.sort_values("R^2", ascending=False, na_position="last")
    display(
        summary.style.format(
            {"PCR": "{:.1%}", "R^2": "{:.4f}"}, 
            na_rep="—" # display missing values such as NaN as "-"
        )
    )

    if plot_PCR:
        for pair_name, (s_metric, c_metric) in SC_metric_pairs.items():
            plot_PCR_pareto(
                valid_data_by_pair[pair_name],
                s_metric=s_metric, c_metric=c_metric, target=target,
                pair_label=SC_metric_pair_labels[pair_name],
                model_name=model_name, pcr=summary.loc[pair_name, "PCR"],
                log_scale=log_scale, markersize=markersize, alpha=alpha,
                cmap=cmap, colorbar_width=colorbar_width,
                colorbar_pad=colorbar_pad,
                tick_labelsize=tick_labelsize,
                annotation_fontsize=annotation_fontsize,
                show_annotation_arrows=show_annotation_arrows,
                figsize=figsize,
            )

    return summary

### Step 3.2 — Run every model

Set `TRAIN_LOSS_MAX` and/or `TEST_LOSS_MAX` to numbers to enable the corresponding loss upper bounds. Use `None` to disable either filter. 

Set `PLOT_PCR=True` to show one Pareto plot per metric pair after each model's summary table.

In [12]:
DATASET_MODELS

{'cifar10': ['resnet18_BN', 'vgg13_BN', 'resnet18_noBN', 'vgg13_noBN'],
 'cifar100': ['wideresnetpostact_BN',
  'wideresnetpostact_noBN',
  'vit_BN',
  'vit_noBN']}

In [ ]:
TRAIN_LOSS_MAX = 0.01
TEST_LOSS_MAX = None  # Example: test_loss <= 2.3 or 4.6



REGRESSION_TARGET = "gap"
PERFORMANCE_HIGHER_BETTER = False
SORT_SUMMARY_TABLE_BY = "PCR"  # None / "PCR" / "R^2"
TICK_LABELSIZE = 8
ANNOTATION_FONTSIZE = 7
SHOW_ANNOTATION_ARROWS = False

PLOT_PCR = False #>>>>>>>>>>>>>>>>>>.<<<<<<<<<<<<<<<<<<<<<<<<
PCR_FIGURESIZE =(5, 4.25)
PCR_MARKERSIZE = 8.5
PCR_ALPHA = 0.80
PCR_CMAP = "YlGnBu"  # "viridis" , "plasma", "coolwarm", "magma"
                # Sequential—values increase from low to high:
                # "cividis", "inferno", "Blues", "Greens", "Reds_r", "YlGnBu_r", "BuPu_r"
PCR_COLORBAR_WIDTH = "1.5%"
PCR_COLORBAR_PAD = "2.5%"

regression_pareto_summaries = {}
for model_name in MODELS:
    dataset = MODEL_DATASETS[model_name]
    print(
        f"\n{'=' * 12} Dataset: {dataset} | Model: {model_name} {'=' * 12}"
    )
    
    regression_pareto_summaries[model_name] = analyze_metric_dataframe(
                    metric_dfs_readytogo[model_name],
                    model_name=model_name,
                    SC_metric_pairs=SC_metric_pairs,
                    SC_metric_pair_labels=SC_metric_pair_labels,
                    target=REGRESSION_TARGET,
                    train_loss_max=TRAIN_LOSS_MAX,
                    test_loss_max=TEST_LOSS_MAX,
                    performance_higher_better=PERFORMANCE_HIGHER_BETTER,
                    sort_summary_table_by=SORT_SUMMARY_TABLE_BY,
                    plot_PCR=PLOT_PCR,
                    figsize=PCR_FIGURESIZE,
                    markersize=PCR_MARKERSIZE,
                    alpha=PCR_ALPHA,
                    cmap=PCR_CMAP,
                    colorbar_width=PCR_COLORBAR_WIDTH,
                    colorbar_pad=PCR_COLORBAR_PAD,
                    tick_labelsize=TICK_LABELSIZE,
                    annotation_fontsize=ANNOTATION_FONTSIZE,
                    show_annotation_arrows=SHOW_ANNOTATION_ARROWS,
    )

resnet18_BN_regression_pareto_summary = regression_pareto_summaries["resnet18_BN"]
vgg13_BN_regression_pareto_summary = regression_pareto_summaries["vgg13_BN"]
resnet18_noBN_regression_pareto_summary = regression_pareto_summaries["resnet18_noBN"]
vgg13_noBN_regression_pareto_summary = regression_pareto_summaries["vgg13_noBN"]
wideresnetpostact_BN_regression_pareto_summary = (regression_pareto_summaries["wideresnetpostact_BN"])
wideresnetpostact_noBN_regression_pareto_summary = (regression_pareto_summaries["wideresnetpostact_noBN"])
vit_BN_regression_pareto_summary = (regression_pareto_summaries["vit_BN"])
vit_noBN_regression_pareto_summary = (regression_pareto_summaries["vit_noBN"])


============ Dataset: cifar10 | Model: resnet18_BN ============
resnet18_BN: 30/48 well-trained points (train_loss <= 0.01).


,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_bayesS_func_det_unsquare_centered,"(bayesS_aniso, func_norm)",30,0.0%,0.7292,"(2.0637e-01, 1.0790e+00)"
pair_input_alpha3_func_det_unsquare_centered,"(inputS_alpha3, func_norm)",30,0.0%,0.9971,"(7.3810e-01, 3.8252e-01)"
pair_adapsam_func_det_unsquare_centered,"(adapSAM, func_norm)",30,0.6%,0.7980,"(1.6500e+00, 7.9211e-01)"
pair_input_alpha2_func_det_unsquare_centered,"(inputS_alpha2, func_norm)",30,1.1%,0.9883,"(2.0627e-01, 4.7358e-01)"
pair_original_sqrt,"(traceH, l2norm)",30,1.4%,0.8368,"(3.8446e-05, 1.0474e-02)"
pair_original,"(traceH, l2norm_sq)",30,1.4%,0.8784,"(4.0482e-05, 1.1183e-04)"
pair_input_alpha1_func_det_unsquare_centered,"(inputS_alpha1, func_norm)",30,1.5%,0.8382,"(1.1287e+00, 1.1474e+00)"
pair_kernel_shared_alpha3_func_det_unsquare_centered,"(kernelS_shared_alpha3, func_norm)",30,1.7%,0.9157,"(3.0651e-01, -4.3394e-02)"
pair_kernel_shared_alpha1_func_det_unsquare_centered,"(kernelS_shared_alpha1, func_norm)",30,1.9%,0.8465,"(8.5081e-01, 1.0073e+00)"



============ Dataset: cifar10 | Model: vgg13_BN ============
vgg13_BN: 34/48 well-trained points (train_loss <= 0.01).


,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_input_alpha3_func_det_unsquare_centered,"(inputS_alpha3, func_norm)",34,1.1%,0.9873,"(8.9633e-01, 5.4500e-01)"
pair_original,"(traceH, l2norm_sq)",34,1.8%,0.7782,"(2.3044e-05, 1.4611e-04)"
pair_original_sqrt,"(traceH, l2norm)",34,1.8%,0.7564,"(2.3992e-05, 1.0597e-02)"
pair_kernel_shared_alpha1_func_det_unsquare_centered,"(kernelS_shared_alpha1, func_norm)",34,2.3%,0.8759,"(1.0289e+00, 9.2546e-01)"
pair_input_alpha1_func_det_unsquare_centered,"(inputS_alpha1, func_norm)",34,2.5%,0.8527,"(1.2198e+00, 1.0164e+00)"
pair_kernel_indep_alpha1_func_det_unsquare_centered,"(kernelS_indep_alpha1, func_norm)",34,3.6%,0.8472,"(1.1313e+00, 9.2919e-01)"
pair_bayesS_func_det_unsquare_centered,"(bayesS_aniso, func_norm)",34,4.0%,0.8001,"(3.0598e-01, 9.2092e-01)"
pair_adapsam_func_det_unsquare_centered,"(adapSAM, func_norm)",34,4.5%,0.8445,"(7.0738e+00, 6.8385e-01)"
pair_kernel_shared_alpha3_func_det_unsquare_centered,"(kernelS_shared_alpha3, func_norm)",34,4.8%,0.9007,"(3.4497e-01, -9.7593e-04)"



============ Dataset: cifar10 | Model: resnet18_noBN ============
resnet18_noBN: 58/60 well-trained points (train_loss <= 0.01).


...

,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_kernel_shared_alpha3_func_det_unsquare_centered,"(kernelS_shared_alpha3, func_norm)",47,13.4%,0.6561,"(4.7386e-01, 9.9945e-01)"
pair_input_alpha3_func_det_unsquare_centered,"(inputS_alpha3, func_norm)",47,14.8%,0.7068,"(3.9443e-01, 1.3492e+00)"
pair_kernel_indep_alpha3_func_det_unsquare_centered,"(kernelS_indep_alpha3, func_norm)",47,21.3%,0.4127,"(2.6189e-01, 9.4397e-01)"
pair_kernel_shared_alpha2_func_det_unsquare_centered,"(kernelS_shared_alpha2, func_norm)",47,22.6%,0.3179,"(8.5990e-02, 9.4347e-01)"
pair_input_alpha1_func_det_unsquare_centered,"(inputS_alpha1, func_norm)",47,23.2%,0.5100,"(4.6497e-01, 1.2562e+00)"
pair_kernel_indep_alpha2_func_det_unsquare_centered,"(kernelS_indep_alpha2, func_norm)",47,26.6%,0.2841,"(-7.5860e-03, 1.0402e+00)"
pair_input_alpha2_func_det_unsquare_centered,"(inputS_alpha2, func_norm)",47,27.9%,0.4788,"(1.2415e-01, 1.1973e+00)"
pair_kernel_shared_alpha1_func_det_unsquare_centered,"(kernelS_shared_alpha1, func_norm)",47,29.0%,0.4518,"(1.4175e+00, 1.2855e+00)"
pair_kernel_indep_alpha1_func_det_unsquare_centered,"(kernelS_indep_alpha1, func_norm)",47,29.8%,0.4090,"(1.5034e+00, 1.3294e+00)"



============ Dataset: cifar100 | Model: vit_BN ============
vit_BN: 48/48 well-trained points (train_loss <= 0.01).


,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_kernel_indep_alpha2_func_det_unsquare_centered,"(kernelS_indep_alpha2, func_norm)",48,1.4%,0.8629,"(7.9443e-02, 1.6763e+00)"
pair_input_alpha2_func_det_unsquare_centered,"(inputS_alpha2, func_norm)",48,1.6%,0.9102,"(1.5942e-01, 1.6220e+00)"
pair_kernel_shared_alpha3_func_det_unsquare_centered,"(kernelS_shared_alpha3, func_norm)",48,1.8%,0.8553,"(1.4118e-01, 2.1097e+00)"
pair_kernel_indep_alpha3_func_det_unsquare_centered,"(kernelS_indep_alpha3, func_norm)",48,1.9%,0.8843,"(2.6810e-01, 1.0723e+00)"
pair_kernel_shared_alpha2_func_det_unsquare_centered,"(kernelS_shared_alpha2, func_norm)",48,2.0%,0.8634,"(5.2438e-02, 1.7084e+00)"
pair_input_alpha3_func_det_unsquare_centered,"(inputS_alpha3, func_norm)",48,3.1%,0.9211,"(4.7076e-01, 1.5461e+00)"
pair_kernel_shared_alpha1_func_det_unsquare_centered,"(kernelS_shared_alpha1, func_norm)",48,11.2%,0.8314,"(1.0929e-01, 2.1796e+00)"
pair_kernel_indep_alpha1_func_det_unsquare_centered,"(kernelS_indep_alpha1, func_norm)",48,11.9%,0.8213,"(1.6692e-01, 2.2336e+00)"
pair_input_alpha1_func_det_unsquare_centered,"(inputS_alpha1, func_norm)",48,12.3%,0.8693,"(1.4152e-01, 1.9177e+00)"



============ Dataset: cifar100 | Model: vit_noBN ============
vit_noBN: 28/48 well-trained points (train_loss <= 0.01).


,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_input_alpha2_func_det_unsquare_centered,"(inputS_alpha2, func_norm)",28,8.6%,0.8996,"(2.3664e-01, 1.5039e-01)"
pair_input_alpha3_func_det_unsquare_centered,"(inputS_alpha3, func_norm)",28,9.0%,0.8657,"(5.5709e-01, 2.2204e-01)"
pair_kernel_shared_alpha1_func_det_unsquare_centered,"(kernelS_shared_alpha1, func_norm)",28,13.5%,0.7418,"(4.3700e-02, 2.5715e+00)"
pair_input_alpha1_func_det_unsquare_centered,"(inputS_alpha1, func_norm)",28,14.4%,0.7403,"(8.3576e-02, 2.6472e+00)"
pair_kernel_indep_alpha1_func_det_unsquare_centered,"(kernelS_indep_alpha1, func_norm)",28,15.1%,0.7470,"(1.3805e-01, 2.6305e+00)"
pair_adapsam_func_det_unsquare_centered,"(adapSAM, func_norm)",28,15.7%,0.7447,"(4.3864e-02, 2.7214e+00)"
pair_kernel_shared_alpha2_func_det_unsquare_centered,"(kernelS_shared_alpha2, func_norm)",28,16.0%,0.7302,"(2.3432e-02, 2.6859e+00)"
pair_bayesS_func_det_unsquare_centered,"(bayesS_aniso, func_norm)",28,16.9%,0.7016,"(-7.1146e-01, 2.9025e+00)"
pair_kernel_indep_alpha3_func_det_unsquare_centered,"(kernelS_indep_alpha3, func_norm)",28,17.1%,0.7012,"(-1.1326e-03, 2.8778e+00)"
